# Core concepts

TAG is a library for building easily multi-level, multi-agent RL agents.

This is possible thanks to the abstractions provided by the `LevelEnv`, the `Hierarchy` and the `LevelAgent` classes that help abstract the hierarchy description and creation.

In this first tutorial we'll describe how the framework works and how these class are used.

### The BaseAgent class

The most general agent in `TAG` is defined by following the `BaseAgent` interface.
It can be imported with:
```python
from tame.hierarchy import BaseAgent
```
This is an abstract base class that defines the interface for all agents in the system. It includes methods for initialization, saving/loading agent state, taking actions based on observations, and training. All methods are abstract and must be implemented by subclasses.
This represent a classic Reinforcement learning algorithm and provides structure for few standard functions:
- **Methods**:
    - `__init__(env: ParallelEnv, args: None | Any = None)`: Initialize the agent.
    - `save_agent(save_path: str | Path, name: None | str = None)`: Save the agent's state.
    - `load_agent(load_path: Path | str, name: str = "trained_model")`: Load the agent's state.
    - `act(observation: Dict[str, np.ndarray])`: Generate actions based on observations.
    - `train(env: Any, log_path: Path | str | None = None, run_name: str | None = None)`: Train the agent.


In our experiments and examples, we use this interface to make sure that the `run_experiment.py` script can train the agent. 
It is usually inside a `BaseAgent` that we instantiate and define the hierarchy.

**Note:** If you don't want to use the `run_experiment.py` script, you don't really need this class. This is mainly to enforce a common basic structure.

### The LevelAgent class

The `LevelAgent` is a specialized agent class designed for hierarchical reinforcement learning, where agents can be part of a multi-level system. 
It inherits from `BaseAgent` and can be imported with:
```python
from tame.hierarchy import LevelAgent
```

Unlike `BaseAgent`, the `LevelAgent` is designed to work within a hierarchical environment level and includes capabilities for inter-level communication. 
Each TAG hierarchical system is composed of multiple `LevelAgents` organized in levels.
This means that every agent in the hierarchy MUST be an instantiation of this class.

In practice, you can wrap any RL agent as a `LevelAgent` just by implementing or wrapping the needed functions. 
An example is PPO from `src/tame/agents/monolithic_ppo.py`.

This class introduces several key attributes:
- **Class Attributes**:
    - `action_space`: Gym Dictionary space describing the action space of the agent.
    - `observation_space`: Gym Dictionary space describing the observation space of the agent.
    - `communication_space`: Gym Dictionary space describing the communication space of the agent. This is optional, as some agents might not need to communicate.
    - `name`: Identifier for the agent (defaults to "base_agent")
    - `torch_compile`: Flag for compiling torch modules (defaults to False)

- **Methods**:
    - `__init__(observation_space, action_space, communication_space, device, name, args, torch_compile)`: Initialize the agent with its spaces and configuration
    - `update_step(global_step, writer)`: Perform one training update during environment steps
    - `store(state, action, reward, done)`: Store transition information in agent memory
    - `act_train(observation, global_step)`: Generate actions during training (with exploration)
    - `seed(seed)`: Set random seeds for reproducibility
    - `comm(observation)`: Generate communication for higher-level agents. This is the φ function described in the paper.

**Important Notes:**
- Unlike `BaseAgent`, `LevelAgent` does not implement a standalone `train()` method as training occurs through `update_step()`.
- Nonetheless, you can implement a `train` function for a `LevelAgent` based agent and run it standalone.
- The `communication_space` allows agents to pass information up the hierarchy
- All methods except `comm()` are abstract and must be implemented by subclasses. The basic `comm` function returns the concatenation of observations from level `l-1`. But this can be overwritten to have learned communication in the hierarchy.
- Agents can be used with or without communication (communication_space can be None)

This class is essential for building hierarchical systems where agents need to interact across different levels of abstraction.

### The LevelEnv class
The `LevelEnv` is the core class for managing hierarchical levels in the framework. 
It wraps and organizes multiple `LevelAgents` a gym environment and handles the hierarchical interactions between levels.
Thanks to the gym interface, any RL algorithm that can work with that interface, can be used on top of a `LevelEnv`.

It can be imported with:
```python
from tame.hierarchy import LevelEnv
```

Each agent in a `LevelEnv` is connected to the agents in levels directly above and below in the hierarchy. These connections are specified through a system of "links":
- **Uplinks**: Define how agents connect to higher levels `{higher_agent: [lower_agents]}`
- **Downlinks**: Define how agents connect to lower levels `{agent: [lower_agents]}`

The class has several key components:
- **Class Attributes**:
    - `agents`: Dictionary of LevelAgents at this level structured as: `{name: agent}`
    - `env`: The lower level environment (This can be a `pettingzoo.ParallelEnv` or another `LevelEnv`)
    - `uplinks/downlinks`: Dictionaries defining hierarchical connections
    - `action_ifreq`: Action frequency relative to lower level steps
    - `observation_spaces/action_spaces`: Spaces describing the action space and the observation spaces of the environement defined by the level.
    - `trace`: Optional trace object for logging episode data

- **Key Methods**:
    - `__init__(agents, uplinks, downlinks, env, ...)`: Initialize the level with its agents and connections
    - `step(action, training)`: Execute one step in the environment
    - `reset()`: Reset the level and return initial observations
    - `make_message(...)`: Create messages for higher level from lower level data (bottom→top)
    - `act(observation, training)`: Generate actions for lower level from agents (top→bottom)
    - `save/load`: Save/load all agents' models

**Important Notes:**
- The `LevelEnv` manages bi-directional communication:
  - Bottom→Top: Lower level observations are processed through `make_message()`
  - Top→Bottom: Higher level actions are distributed through `act()`
- Training occurs through `update_step()` which is called during environment steps
- Each level can operate at different time scales using `action_ifreq`
- The environment supports optional tracing for logging episode data
- Agents can communicate upward using their `comm()` function (the φ function in the paper)

This class is the backbone of implementing hierarchical RL systems in TAG, as it provides the structure for multi-level agent interactions and training.

## The Hierarchy
In practice, the whole hierarchy can be instantiated by using configurations for the agents and the levels passed directly to the `Hierarchy` constructor. This class manages the entire hierarchical structure and can be imported with:
```python
from tame.hierarchy import Hierarchy
```

The configuration is done through two dataclasses:
- **AgentConfig**: Defines individual agent properties
  - `name`: Agent identifier
  - `communication_space`: Space for communication with higher levels
  - `agent_class`: The LevelAgent implementation to use
  - `agent_kwargs`: Additional arguments for agent initialization
  - `device`: Device for the agent (CPU/GPU)

- **LevelConfig**: Defines a complete level in the hierarchy
  - `name`: Level identifier
  - `agents`: List of AgentConfig objects
  - `env`: Environment (ParallelEnv or LevelEnv)
  - `uplinks/downlinks`: Communication structure
  - `action_frequency`: Timing of actions relative to base level
  - `concat_obs`: Whether to concatenate observations from below with actions from above as observations for the agents.
  - `action_space`: Action space for the level

The `Hierarchy` class provides several key functionalities:

- **Core Methods**:
  - `add_level_config(level_cfg)`: Add a new level using configuration
  - `reset()`: Reset entire hierarchy
  - `step(action)`: Step through hierarchy from top to bottom
  - `act(observation)`: Process observations up and actions down
  - `save(path)/load(path)`: Save/load all levels

- **Visualization Methods**:
  - `tree()`: Get hierarchical structure as dictionary
  - `print_tree()`: Print ASCII visualization of hierarchy
  - `print_hierarchy_details()`: Print comprehensive structure details

**Information Flow**:
1. Bottom-up pass:
   - Environment observations enter at the bottom level
   - Each level processes observations through `make_message()`. In this step, the `comm()` function of the agents is called to generate their messages. Then the `make_message()` function from the `LevelEnv` in which the agents are groups these messages and routes them depending on the connections defined in `uplinks`.
   - Messages become observations for the next level up
   - Process continues until reaching top level

2. Top-down pass:
   - Top level generates initial actions. This is done by calling the `act()` function of the agents in the top level. The `LevelEnv` then routes the actions to the level below according to the connections defined in `downlinks`.
   - Each level receives actions from above
   - Actions are combined with bottom-up information
   - Process continues until reaching environment level

**Important Notes:**
- The hierarchy processes information in two passes:
  1. Bottom-up: Observations travel up through `make_message()` and `comm()`
  2. Top-down: Actions travel down through `act()`
- Each level can operate at different time scales using `action_frequency`
- Agents can be configured to concatenate observations from higher levels
- The hierarchy handles saving/loading of all agents across all levels
- Visualization tools help understand and debug the structure

The `Hierarchy` class is the top-level manager for complex hierarchical systems, handling the coordination between levels and providing tools for configuration, operation, and visualization.

Example usage:
```python
# Configure an agent
agent_cfg = AgentConfig(
    name="agent1",
    communication_space=GymDict(...),
    agent_class=MyLevelAgent,
    agent_kwargs={"learning_rate": 0.001},
    device=torch.device("cuda")
)

# Configure a level
level_cfg = LevelConfig(
    name="level1",
    agents=[agent_cfg],
    env=my_env,
    uplinks={"higher_agent": ["agent1"]},
    downlinks={"agent1": ["env_agent1", "env_agent2"]},
    action_frequency=5
)

# Create and configure hierarchy
hierarchy = Hierarchy()
hierarchy.add_level_config(level_cfg)
```

# Spaces correspondance

In TAG, both agents and levels have action and observation spaces. We describe here the correpondance between them.

Each level has inputs and output defined by an `observation_space` and `action_space`. These are the equivalent of the observation and action spaces of a gym environment. 
- The `observation_space` describes the observations that the level sends to the agents acting on it. 
- The `action_space` describe the actions that this level expects from the agents acting on it.

Each agent in a level has three spaces:
- `action_space`: defines the actions that the agent produces, to be sent to the level below.
- `communication_space`: defines the communication messages that the agent produces, to be sent to the level above. The collection of the `communication_space` of all the agents in a level, correponds to the `observation_space` of the level.
- `observation_space`: The observations that the agent expects. This depends on the `concat_obs` variable of the level in which the agent is. If `concat_obs=True`, the observation of an agent corresponds to the concatenation of the observations from the level below and the actions from the level above. If `concat_obs=False`, the observation of the agent are only the actions coming from the agents in the level above.

*Note that when defining a hierarchy, you only need to specify the `communication_space` for each agent, and the `action_space` for the levels. The rest are inferred automatically by the hierarchy builder.*

A representation of the information flow between the levels is shown below. More details can be found in the paper: [paper_link]

<img src="./images/information_flow.png" alt="Information Flow" width="600"/>